In [ ]:
#@title **Grain Shape Analysis Notebook**

#@markdown **Implementation of the Manuscript:**
#@markdown This notebook corresponds to the implementation of the manuscript:
#@markdown
#@markdown Escribano Leiva, D., Kuncar Medina, C., & Montalva, G. (2025). Using a smartphone device to quantify particle size and shape descriptors. *Acta Geotechnica*, *20*(10), 5277–5295. https://doi.org/10.1007/s11440-025-02676-x

#@markdown ---
#@markdown [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camedinak24/fastGRAINS25/blob/main/FastGrains.ipynb)

#@title **Author Information**

#@markdown **Daniella Escribano Leiva, PhD**
#@markdown - Department of Civil Engineering, Universidad de Concepción, Chile
#@markdown - Fondecyt Iniciación Nº11241067, ANID, Chile
#@markdown - Email: describano@udec.cl
#@markdown - ORCID: 0000-0003-2014-9008

#@markdown **Carlos Kuncar Medina, MSc.**
#@markdown - Department of Civil Engineering, Universidad de Concepción, Chile
#@markdown - Email: camedina2017@udec.cl

#@markdown **Gonzalo Montalva, PhD**
#@markdown - Department of Civil Engineering, Universidad de Concepción, Chile
#@markdown - EASER Project ACT240044, ANID, Chile
#@markdown - Email: gmontalva@udec.cl
#@markdown - ORCID: 0000-0001-8598-7120

#@markdown ---

#@title **Abstract Summary**

#@markdown This notebook provides a procedure to obtain particle size and shape descriptors through a 2D image processing algorithm using a smartphone device. The open-source tool is automatic and requires only a simple initial calibration of the camera lens. The output consists of a spreadsheet with particle size and shape descriptors for each grain, as well as their average values for the entire sample. The tool benefits the geotechnical community due to the strong link between particle morphology and mechanical properties. An application is included where the script is used to obtain particle shape descriptors of a sand with known critical state parameters, evaluating them through particle shape predictive models. The results indicate that analyzing images with a minimum of 30 particles provides a good match with experimental data, highlighting the advantages of the tool as a first estimate and as a complement to a full experimental program.

#@markdown ---

#@title **Notebook Guide**

#@markdown This notebook is organized into several sections, each with detailed instructions and explanations. Please run the cells by clicking on the top left corner of each cell (on the "Play" button ▶) to ensure that the analysis runs smoothly.

#@markdown **Note:** This notebook is optimized for **Google Colab** and should be run using this platform.

#@markdown ---

In [ ]:
#@title **1. Install software**
#@markdown 🔙⬅ Click the "Play" button on the left to run this cell and install the necessary libraries for this Google Colab notebook. All installations are performed on Google Colab and not on your local machine.

# Optional: uncomment to hide output
# %%capture

# Check if repository already exists to avoid duplicate cloning
import os
os.environ['PYTHONHASHSEED'] = '0'
if not os.path.exists('fastGRAINS25'):
    print("📥 Cloning repository...")
    !git clone https://github.com/ckuncarm/fastGRAINS25.git
else:
    print("📂 Repository already exists. Updating...")
    %cd fastGRAINS25
    !git pull
    %cd ..

# Navigate to repository directory
%cd fastGRAINS25/

# Install dependencies with progress feedback
print("\n📦 Installing dependencies...")
!pip install -qr FastGRAINS/requirements.txt
# !pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu118
# Download model weights with verification
print("\n📥 Downloading model weights...")
import os.path
if not os.path.isfile('FastSAM.pt'):
    !wget -q --show-progress https://huggingface.co/spaces/An-619/FastSAM/resolve/main/weights/FastSAM.pt
    print("✅ Model weights downloaded successfully")
else:
    print("✅ Model weights already exist")

# Check for common errors and verify installation
try:
    print(f"\n✅ Installation successful!")
    print("\n🔁 Restart the kernel by executing cell 2 to terminate...\n")
except Exception as e:
    print(f"\n❌ Installation error: {e}")
    print("Please run this cell again if you encounter any issues.")

In [ ]:
#@title **2. ⚠️Restart kernel to apply changes**
#@markdown  IMPORTANT: Run this cell after installation to restart the kernel
#@markdown
#@markdown This restart is necessary for all installed packages to be properly loaded.
#@markdown After the restart completes, continue with cell #3.
#@markdown
#@markdown ⚠️ NOTE: This cell will show an error message "Kernel died, restarting" - this is NORMAL and EXPECTED.
#@markdown The error indicates the kernel is successfully restarting.

print("\n🔁 Restarting kernel to apply changes...")
print("⚠️ Please wait until the restart completes before running the next cell")
print("⚠️ You will see an error message - this is normal and part of the restart process")
print("⚠️ After restart completes, continue with cell #3 to set up the environment")
exit()


🔁 Restarting kernel to apply changes...
⚠️ Please wait until the restart completes before running the next cell
⚠️ You will see an error message - this is normal and part of the restart process
⚠️ After restart completes, continue with cell #3 to set up the environment


------

In [ ]:
#@title **3. Creation of directories and loading of functions**
#@markdown Enter the material name, e.g., **BBSAND** ⬇️
#@markdown
#@markdown ⚠️ Make sure the kernel restart from cell #2 has completed before running this cell

# Check current directory and set up environment
import time
import os
current_dir = os.getcwd()
if current_dir.endswith('fastGRAINS25'):
    print("✅  fastGRAINS25 directory loaded successfully")
else:
    try:
        %cd "fastGRAINS25/"
        print("✅  fastGRAINS25 directory loaded successfully")
    except:
        print("⚠️ Could not change to fastGRAINS25 directory. Make sure to run cells 1 and 2 first.")
        # Create the directory if it doesn't exist
        if not os.path.exists("fastGRAINS25"):
            os.makedirs("fastGRAINS25")
            %cd "fastGRAINS25/"
            print("✅ Created and changed to fastGRAINS25 directory")

print("Initializing FastGRAINS25...")

# Step 1: Import libraries
print("Step 1/6: Importing libraries...")
import sys
import zipfile
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
import glob

# Step 2: Configure paths
print("Step 2/6: Configuring paths...")
sys.path.append('./FastGRAINS')  # Add FastGRAINS to system path
sys.path.append('./FastSAM')     # Add FastSAM to system path

# Step 3: Import core modules
print("Step 3/6: Loading core modules...")

from image_io import create_directories, plot_image, upload_image, upload_images, plot_grids
from constants import set_material_name, IOU_THRESHOLD, DEVICE, IMAGE_SIZE, RESCALE, PAD, PCD_Target
from preprocessing import preprocess_image, postprocess_image, resize_image_keep_aspect
from background_removal import remove_background, invert_image
from segmentation import run_inference, load_annotations
from feature_extraction import get_bboxes, get_centroids, calculate_grain_metrics, save_metrics

# Step 4: Set material name
print("Step 4/6: Setting material name...")
material_name = 'BBSAND'  #@param {type:"string"}
set_material_name(material_name)

# Step 5: Import additional modules
print("Step 5/6: Loading additional modules...")
from postprocessing import segment_and_store, remove_background_for_single_grain, resize_grains, grid_grains, save_grains
from utilities import resize_image, process_binary_image

import gc

# from constants import IMAGES_DIR, OUTPUT_DIR

from constants import OUTPUT_DIR
IMAGES_DIR = "/INPUT"
IMAGES_DIR = os.path.join(IMAGES_DIR, material_name)

# Step 6: Set up directories
print("Step 6/6: Setting up directories...")
directories = [IMAGES_DIR, OUTPUT_DIR]
for directory in directories:
    if not os.path.exists(directory):
        os.makedirs(directory)

# Final status message
print("\n✅ FastGRAINS25 environment initialized successfully")
print(f"🔍 Current material: {material_name}")
print("📊 Ready to process grain images")

In [ ]:
#@title **4. Images Upload**

#@markdown Use this cell to upload images for analysis.
#@markdown
#@markdown This cell allows you to upload images for analysis. You have three options:
#@markdown
#@markdown ### Upload Your Own Images
#@markdown 1. First, make sure you've set a material name in the text field.
#@markdown 2. Click the **Upload Images** button to open a file browser.
#@markdown 3. Select one or more image files (supported formats: .png, .jpg, .jpeg, .tif, .tiff).
#@markdown
#@markdown ### Use Example Images
#@markdown If you prefer to use pre-loaded example images:
#@markdown 1. Make sure you've set a material name in the text field.
#@markdown 2. Click the **Use Examples** button to load example images from the FastGRAINS Examples directory.
#@markdown 3. The example images will be displayed as thumbnails.
#@markdown
#@markdown ### Clear Images
#@markdown To remove all loaded images and start fresh, click the **Clear All Images** button.
#@markdown
#@markdown #### Important Notes:
#@markdown - You must set a material name before uploading or using example images
#@markdown - ⚠️ If you want to process multiple images, consider splitting the processing into batches of approximately 5 images. Large images can exhaust the resources offered by Google Colab (RAM).
#@markdown
import shutil
import glob
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output
import PIL
from PIL import Image

set_material_name(material_name)  # Set the material name dynamically

from constants import IMAGES_DIR, OUTPUT_DIR

# Ensure directories exist
if not os.path.exists(IMAGES_DIR):
    os.makedirs(IMAGES_DIR)

VALID_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')

# Global variable to track image paths
images_path = []

def set_material_name(name):
    """Set the material name and update global variable"""
    global material_name
    material_name = name
    print(f"Material name set to: {material_name}")
    return material_name

def refresh_images_path():
    """Refresh the list of valid images in the output directory"""
    global images_path
    images_path = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*')))
    images_path = [f for f in images_path if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS]

    # Display preview of images
    preview_images()

    print(f"Finished loading images. Total images: {len(images_path)}")
    for img in images_path:
        print(" - " + os.path.basename(img))

    return images_path

def preview_images():
    """Display thumbnails of currently loaded images"""
    if not images_path:
        return

    # Create thumbnail grid
    thumbs = []
    for i, img_path in enumerate(images_path[:12]):  # Show up to 12 thumbnails
        try:
            img = Image.open(img_path)
            img.thumbnail((150, 150))
            img_widget = widgets.Image(
                value=img._repr_png_(),
                format='png',
                width=150,
                height=150,
                layout=widgets.Layout(margin='2px')
            )
            label = widgets.Label(os.path.basename(img_path))
            thumbs.append(widgets.VBox([img_widget, label]))
        except Exception as e:
            print(f"Error loading thumbnail for {img_path}: {e}")

    # Display grid of thumbnails
    if thumbs:
        grid = widgets.HBox(thumbs)
        display(grid)

def handle_upload():
    """Handle file uploads from user's computer"""
    if not material_name:
        print("❌ Please set material name first!")
        return

    # Clear existing files
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Please select files to upload:")
    uploaded = files.upload()  # Opens the file chooser

    valid_files = 0
    errors = 0

    for i, (name, content) in enumerate(uploaded.items(), start=1):
        ext = os.path.splitext(name)[1].lower()
        if ext not in VALID_EXTENSIONS:
            print(f"⚠️ Skipping invalid file type: {name}")
            continue

        try:
            # Verify it's a valid image by attempting to open it
            img_data = BytesIO(content)
            Image.open(img_data).verify()

            # Save the file with the material name prefix
            new_name = f"{material_name}_{i}{ext}"
            dest = os.path.join(OUTPUT_DIR, new_name)
            with open(dest, 'wb') as f:
                f.write(content)
            valid_files += 1
        except Exception as e:
            print(f"❌ Error processing {name}: {e}")
            errors += 1

    refresh_images_path()
    print(f"✅ Upload complete! Loaded {valid_files} valid images. Errors: {errors}")

def handle_examples():
    """Load example images from the FastGRAINS examples directory"""
    if not material_name:
        print("❌ Please set material name first!")
        return

    # Clear existing files
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    examples_dir = os.path.join('./FastGRAINS', 'Examples')

    if not os.path.exists(examples_dir):
        print(f"❌ Examples directory not found: {examples_dir}")
        return

    example_images = [f for f in os.listdir(examples_dir)
                     if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS]

    if not example_images:
        print(f"❌ No example images found in: {examples_dir}")
        return

    for i, img in enumerate(example_images, start=1):
        ext = os.path.splitext(img)[1].lower()
        new_name = f"{material_name}_example_{i}{ext}"
        try:
            shutil.copy(os.path.join(examples_dir, img),
                      os.path.join(OUTPUT_DIR, new_name))
        except Exception as e:
            print(f"❌ Error copying example {img}: {e}")

    refresh_images_path()
    print(f"✅ Success! Loaded {len(images_path)} example images.")

def clear_images():
    """Clear all images from the output directory"""
    global images_path
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    images_path = []
    clear_output(wait=True)
    print("🧹 All images cleared.")
    display_upload_interface()

def display_upload_interface():
    """Display the upload interface with buttons"""
    # Set the material name if provided
    if material_name:
        set_material_name(material_name)

    # Create buttons
    upload_button = widgets.Button(
        description="Upload Images",
        button_style='primary',
        icon='upload',
        layout=widgets.Layout(width='200px', height='40px')
    )

    examples_button = widgets.Button(
        description="Use Examples",
        button_style='info',
        icon='image',
        layout=widgets.Layout(width='200px', height='40px')
    )

    clear_button = widgets.Button(
        description="Clear All Images",
        button_style='danger',
        icon='trash',
        layout=widgets.Layout(width='200px', height='40px')
    )

    status_output = widgets.Output()

    # Set up button callbacks
    def on_upload_click(b):
        with status_output:
            clear_output(wait=True)
            handle_upload()

    def on_examples_click(b):
        with status_output:
            clear_output(wait=True)
            handle_examples()

    def on_clear_click(b):
        clear_images()

    upload_button.on_click(on_upload_click)
    examples_button.on_click(on_examples_click)
    clear_button.on_click(on_clear_click)

    # Create material name input widget for dynamic updates
    material_input = widgets.Text(
        value=material_name,
        description='Material:',
        placeholder='Enter material name',
        layout=widgets.Layout(width='300px')
    )

    def on_material_change(change):
        set_material_name(change['new'])

    material_input.observe(on_material_change, names='value')

    # Display widgets
    display(widgets.VBox([
        material_input,
        widgets.HBox([upload_button, examples_button, clear_button]),
        status_output
    ]))

    # Initial refresh to show any preexisting images
    with status_output:
        refresh_images_path()
        print(f"📸 Current images ready: {len(images_path)} images")

# Fix missing import
from io import BytesIO

# Display the upload interface
display_upload_interface()

In [ ]:
#@title **5. Segmentation batch process**
#@markdown This cell performs the segmentation and processing of uploaded images.
#@markdown
#@markdown ⚠️ This process may take several minutes depending on the number and size of images.
#@markdown The progress indicator will update as each image is processed.

import gc
import shutil
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import numpy as np
import time
from tqdm.notebook import tqdm
import ipywidgets as widgets

# Import required functions
from shape_metrics import compute_shape_metrics, compute_metrics_for_all_keys, resize_grains_to_PCD
from visualization import plot_grain, interactive_plot, draw_circles_on_grain

# Progress display setup
progress_output = widgets.Output()
display(progress_output)

def process_image(image_path, progress_bar=None):
    """Process a single image through the grain analysis pipeline"""
    with progress_output:
        print(f"Processing: {os.path.basename(image_path)}")

    # Background Removal (suppress output with semicolon)
    resized_path, black_background_path, white_background_path, avg_background_path, reduction_factor = remove_background(image_path, 2048, OUTPUT_DIR, RESCALE);

    # Segmentation
    annotations = run_inference(black_background_path, IMAGE_SIZE, IOU_THRESHOLD)

    # Load Annotations
    annotations = load_annotations(annotations, min_major_axis_length=int(10 / reduction_factor))
    gc.collect()

    # Feature Extraction
    bboxes = get_bboxes(annotations, PAD)
    centroids = get_centroids(annotations)

    # Post-processing
    grains = segment_and_store(annotations, bboxes, black_background_path, PAD)

    # Remove background for each grain
    grains = {grain_id: {**grain_data, "grain_rgb": remove_background_for_single_grain(grain_data, "black")}
              for grain_id, grain_data in grains.items()}

    # Process grains
    resize_grains(grains, reduction_factor)
    calculate_grain_metrics(grains, reduction_factor, image_path)
    save_grains(grains, image_path, OUTPUT_DIR)
    gc.collect()

    # Resize grains to 1200 px of PCD
    resize_grains_to_PCD(grains, reduction_factor, PCD_Target=1200)

    # Prefix grain_id with image_path to ensure uniqueness and add image_path to each grain's data
    grains = {f"{os.path.basename(image_path)}_{grain_id}": {**grain_data, "image_path": image_path}
              for grain_id, grain_data in grains.items()}

    gc.collect()
    plt.close('all')

    if progress_bar:
        progress_bar.update(1)

    return grains

def main():
    """Process all images in the images_path list"""
    # Check if images are available
    if not images_path:
        with progress_output:
            print("❌ No images found. Please upload images using cell #3 first.")
            return {}

    all_grains = {}
    start_time = time.time()

    with progress_output:
        clear_output(wait=True)
        print(f"🔍 Starting grain segmentation of {len(images_path)} images...")
        progress_bar = tqdm(total=len(images_path), desc="Processing images")

    # Process each image
    for image_path in images_path:
        try:
            grains = process_image(image_path, progress_bar)
            all_grains.update(grains)
        except Exception as exc:
            with progress_output:
                print(f'❌ Error processing {os.path.basename(image_path)}: {exc}')

    # Renumber the dictionary so that grain_id is enumerated from 0
    renumbered_grains = {i: grain_data for i, (grain_id, grain_data) in enumerate(all_grains.items())}

    # Final cleanup
    gc.collect()

    # Completion message
    elapsed_time = time.time() - start_time
    with progress_output:
        clear_output(wait=True)
        print(f"✅ Processing complete! Elapsed time: {elapsed_time:.1f} seconds")
        print(f"🧮 Total grains detected: {len(renumbered_grains)}")
        print(f"📊 Ready for analysis")

    return renumbered_grains

# Execute the main function (wrapped in %%capture to suppress unnecessary output)
concatenated_grains = main()
grains = concatenated_grains.copy()
filtered_grains = grains.copy()

In [ ]:
#@title **6. Select and process grains**
#@markdown 🖱️ Select valid grains and remove segmentation errors

#@markdown **Instructions:**
#@markdown **Instructions:**
#@markdown 1. Use this cell to clean up your segmentation results by removing:
#@markdown    - Improperly segmented grains (incomplete or cut-off particles)
#@markdown    - Overlapping or touching particles
#@markdown    - Any artifacts or non-grain objects detected during segmentation
#@markdown
#@markdown 2. **How to use:**
#@markdown    - Click individual grains to select (✅) or deselect (⬜) them
#@markdown    - Use `Click` to select multiple grains at once
#@markdown
#@markdown 3. Press the ✅ **GENERATE GRIDS** button when you've removed all problematic grains
#@markdown
#@markdown  **Note:** Higher quality results require careful selection - take time to review each grain

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import os
import numpy as np
import io
import matplotlib.pyplot as plt
from functools import partial
import gc
from constants import IMAGES_DIR

# Global variables to store grain data for processing
filtered_grains = {}

def create_rgb_and_binary_grids(grains, output_rgb_path=None, output_binary_path=None, background_color='black', generate_grids=True):
    """
    Creates an interactive UI for selecting grains and optionally generating image grids.

    Args:
        grains: Dictionary of grain data
        output_rgb_path: Path to save RGB grid image (optional if generate_grids=False)
        output_binary_path: Path to save binary grid image (optional if generate_grids=False)
        background_color: Color for RGB grid background ('black' or 'white')
        generate_grids: Boolean flag to control whether to generate grid images

    Returns:
        Dictionary containing only the selected grains
    """
    global filtered_grains  # Access the global variable
    if not grains:
        print("❌ No grains available for processing! Please run the segmentation first.")
        return {}
    filtered_grains = grains.copy()  # Initialize with a copy of the input grains

    # Build list of images with IDs and sizes
    image_list = [
        {
            'id': grain_id,
            'img_rgb': grain_data.get('grain_rs', None),
            'img_bin': grain_data.get('grain_bin_rs', None),
            'size': (grain_data.get('grain_rs').width * grain_data.get('grain_rs').height
                    if grain_data.get('grain_rs') else 0),
            'metrics': grain_data.get('metrics', {})
        }
        for grain_id, grain_data in grains.items()
        if grain_data.get('grain_rs')
    ]

    # Sort images by size (largest first)
    image_list.sort(key=lambda x: x['size'], reverse=True)

    if not image_list:
        print("❌ No valid grain images found!")
        return {}

    # Progress indicator
    progress = widgets.IntProgress(
        value=0,
        min=0,
        max=len(image_list),
        description='Loading:',
        bar_style='info',
        orientation='horizontal'
    )
    display(progress)

    # Create widgets for filter controls
    min_size_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=100,
        step=5,
        description='Min Size %:',
        layout=widgets.Layout(width='300px')
    )

    max_size_slider = widgets.IntSlider(
        value=100,
        min=0,
        max=100,
        step=5,
        description='Max Size %:',
        layout=widgets.Layout(width='300px')
    )

    select_all_btn = widgets.Button(
        description='Select All',
        button_style='info',
        layout=widgets.Layout(width='120px')
    )

    deselect_all_btn = widgets.Button(
        description='Deselect All',
        button_style='warning',
        layout=widgets.Layout(width='120px')
    )

    filter_output = widgets.Output()
    display(widgets.HBox([min_size_slider, max_size_slider, select_all_btn, deselect_all_btn]))
    display(filter_output)

    # Create containers for image widgets
    image_container = widgets.Output()
    display(image_container)

    # Container for all checkboxes and their data
    checkboxes = []

    # Function to update the image display based on filters
    def update_image_display():
        if not checkboxes:
            return

        with image_container:
            clear_output(wait=True)

            # Get size values for filtering
            min_size_pct = min_size_slider.value / 100
            max_size_pct = max_size_slider.value / 100

            # Find the maximum size for percentage calculation
            max_size = max(item['data']['size'] for item in checkboxes)

            # Filter items based on size
            visible_items = []
            for item in checkboxes:
                size_pct = item['data']['size'] / max_size
                if min_size_pct <= size_pct <= max_size_pct:
                    item['widget'].layout.display = 'block'
                    visible_items.append(item['widget'])
                else:
                    item['widget'].layout.display = 'none'

            # Create a grid layout for the visible items
            rows = []
            current_row = []
            for i, item in enumerate(visible_items):
                current_row.append(item)
                if len(current_row) == 4 or i == len(visible_items) - 1:
                    rows.append(widgets.HBox(current_row))
                    current_row = []

            grid = widgets.VBox(rows)
            display(grid)

            # Show count of visible items
            print(f"Showing {len(visible_items)} of {len(checkboxes)} grains")

    # Function to handle select/deselect all
    def select_all(b):
        for item in checkboxes:
            item['checkbox'].value = True

    def deselect_all(b):
        for item in checkboxes:
            item['checkbox'].value = False

    select_all_btn.on_click(select_all)
    deselect_all_btn.on_click(deselect_all)

    # Connect filter controls to update function
    min_size_slider.observe(lambda x: update_image_display(), names='value')
    max_size_slider.observe(lambda x: update_image_display(), names='value')

    # Process each image and create widgets
    all_widgets = []
    for i, item in enumerate(image_list):
        img_rgb = item['img_rgb']
        img_bin = item['img_bin']

        # Create thumbnails
        img_rgb_thumbnail = img_rgb.copy()
        img_rgb_thumbnail.thumbnail((180, 180))

        # Handle binary image (create white image if None)
        if img_bin:
            img_bin_thumbnail = img_bin.copy()
        else:
            img_bin_thumbnail = Image.new('RGB', img_rgb.size, 'white')
        img_bin_thumbnail.thumbnail((180, 180))

        # Combine thumbnails side by side
        combined = Image.new(
            'RGB',
            (img_rgb_thumbnail.width + img_bin_thumbnail.width + 10,  # Add some padding
             max(img_rgb_thumbnail.height, img_bin_thumbnail.height))
        )
        combined.paste(img_rgb_thumbnail, (0, 0))
        combined.paste(img_bin_thumbnail, (img_rgb_thumbnail.width + 10, 0))

        # Convert to bytes for widget
        buffer = io.BytesIO()
        combined.save(buffer, format='PNG')
        buffer.seek(0)

        # Create a checkbox and an image widget
        checkbox = widgets.Checkbox(
            value=True,
            description=f"Grain {item['id']}",
            indent=False
        )

        # Extract some metrics if available
        metrics_text = ""
        if item['metrics']:
            try:
                metrics_text = f"AR: {item['metrics'].get('aspect_ratio', 'N/A'):.2f} | "
                metrics_text += f"R: {item['metrics'].get('roundness', 'N/A'):.2f} | "
                metrics_text += f"S: {item['metrics'].get('sphericity', 'N/A'):.2f}"
            except (TypeError, AttributeError, ValueError):
                metrics_text = "Metrics N/A"

        # Create a label with metrics
        metrics_label = widgets.HTML(
            value=f"<small>{metrics_text}</small>"
        )

        img_widget = widgets.Image(
            value=buffer.getvalue(),
            format='png',
            width=360,
            height=180
        )

        # Combine checkbox, metrics and image in a box
        grain_box = widgets.VBox([
            widgets.HBox([checkbox, metrics_label]),
            img_widget
        ], layout=widgets.Layout(
            margin='5px',
            border='1px solid #ddd',
            padding='5px'
        ))

        # Add to tracking containers
        checkboxes.append({
            'checkbox': checkbox,
            'widget': grain_box,
            'data': {
                'id': item['id'],
                'img_rgb': img_rgb,
                'img_bin': img_bin,
                'size': item['size']
            }
        })

        # Update progress
        progress.value = i + 1

    # Initial display update
    update_image_display()

    # Create and display the action button - adapt label based on generate_grids
    button_description = '✅ GENERATE GRIDS' if generate_grids else '✅ FILTER GRAINS'
    action_button = widgets.Button(
        description=button_description,
        button_style='success',
        layout=widgets.Layout(width='200px', height='60px', font_weight='bold'),
        style={'font_size': '16px'}
    )
    display(action_button)

    # Output area for generation results
    action_output = widgets.Output()
    display(action_output)

    def on_action_click(b):
        global filtered_grains  # Access the global variable

        with action_output:
            clear_output(wait=True)
            print("🔄 Processing selected grains...")

            # Gather selected items from the checkboxes
            selected = [item for item in checkboxes if item['checkbox'].value]

            if not selected:
                print("⚠️ No grains selected! Please select at least one grain.")
                return {}

            print(f"✓ Found {len(selected)} selected grains")

            # Sort by size (largest first)
            selected.sort(key=lambda x: x['data']['size'], reverse=True)

            # Get the selected IDs
            selected_ids = [item['data']['id'] for item in selected]

            # Update the global filtered_grains with only the selected keys
            filtered_grains = {k: grains[k] for k in selected_ids if k in grains}
            print(f"✓ Filtered {len(filtered_grains)} grains for processing")

            # If generate_grids is False, we're done here
            if not generate_grids:
                print(f"🎉 Selected {len(selected_ids)} grains have been filtered!")
                return filtered_grains

            # If generate_grids is True, continue with grid generation
            try:
                # Make sure we have paths for the grid images
                if not output_rgb_path or not output_binary_path:
                    print("❌ Error: Output paths are required when generate_grids=True")
                    return filtered_grains

                # Calculate dimensions
                num_cols = min(len(selected), 6)  # Max 6 columns
                cell_size = max(
                    max(item['data']['img_rgb'].width, item['data']['img_rgb'].height)
                    for item in selected
                )

                # Ensure minimum cell size for visibility
                cell_size = max(cell_size, 200)

                # Calculate rows needed
                num_rows = (len(selected) + num_cols - 1) // num_cols

                # Set background colors
                bg_color = (0, 0, 0) if background_color == 'black' else (255, 255, 255)

                # Create new images for grids
                grid_img = Image.new('RGB', (cell_size * num_cols, cell_size * num_rows), bg_color)
                grid_binary_img = Image.new('RGB', (cell_size * num_cols, cell_size * num_rows), (255, 255, 255))

                # Progress for grid generation
                grid_progress = widgets.IntProgress(
                    value=0,
                    min=0,
                    max=len(selected),
                    description='Building:',
                    bar_style='info',
                    orientation='horizontal'
                )
                display(grid_progress)

                # Paste each image into the grid
                for idx, item in enumerate(selected):
                    row, col = divmod(idx, num_cols)
                    x, y = col * cell_size, row * cell_size

                    # Get image data
                    rgb_img = item['data']['img_rgb']
                    bin_img = item['data']['img_bin']

                    # Paste RGB image with proper centering
                    rgb_bg = Image.new('RGB', (cell_size, cell_size), bg_color)
                    rgb_bg.paste(
                        rgb_img,
                        ((cell_size - rgb_img.width) // 2,
                         (cell_size - rgb_img.height) // 2)
                    )
                    grid_img.paste(rgb_bg, (x, y))

                    # Paste binary image with proper centering
                    bin_bg = Image.new('RGB', (cell_size, cell_size), (255, 255, 255))
                    if bin_img:
                        bin_bg.paste(
                            bin_img,
                            ((cell_size - bin_img.width) // 2,
                             (cell_size - bin_img.height) // 2)
                        )
                    grid_binary_img.paste(bin_bg, (x, y))

                    # Update progress
                    grid_progress.value = idx + 1

                # Save the grid images
                output_dir = os.path.dirname(output_rgb_path)
                os.makedirs(output_dir, exist_ok=True)
                grid_img.save(output_rgb_path)
                grid_binary_img.save(output_binary_path)

                # Display preview of the grids
                fig, axes = plt.subplots(1, 2, figsize=(12, 6))
                axes[0].imshow(np.array(grid_img))
                axes[0].set_title("RGB Grid")
                axes[0].axis('off')

                axes[1].imshow(np.array(grid_binary_img))
                axes[1].set_title("Binary Grid")
                axes[1].axis('off')

                plt.tight_layout()
                plt.show()

                print(f"💾 Grids successfully saved to:")
                print(f"📁 RGB Grid: {output_rgb_path}")
                print(f"📁 Binary Grid: {output_binary_path}")
                print(f"🎉 Selected {len(selected_ids)} grains have been processed and saved!")

            except Exception as e:
                print(f"❌ Error generating grids: {e}")
                import traceback
                traceback.print_exc()

            # Clean up to free memory
            gc.collect()

            return filtered_grains

    action_button.on_click(on_action_click)
    return filtered_grains

# Example usage with new parameters
def run_grain_selection(grains_dict=None, material_name=None, generate_grids=True):
    """Run the grain selection workflow and return filtered_grains"""
    if grains_dict is None:
        print("⚠️ No grains available. Please run segmentation first.")
        return None  # Explicitly return None for clarity

    # Define output paths
    rgb_path = os.path.join(IMAGES_DIR, 'grid_rgb.png')
    bin_path = os.path.join(IMAGES_DIR, 'grid_bin.png')

    # Run the selection UI (this updates the global filtered_grains)
    create_rgb_and_binary_grids(grains_dict, rgb_path, bin_path, background_color='black', generate_grids=generate_grids)

    # Return the global filtered_grains variable
    return filtered_grains  # Now returns the global state

filtered_grains = run_grain_selection(grains, material_name, generate_grids=False)

In [ ]:
#@title **7. Find the optimal parameters for calculating roundness**
#@markdown ⬇️ Use the interactive buttons to adjust the parameters and visualise their effects. ⬇️

#@markdown **Purpose & Functionality**
#@markdown This interactive tool helps optimize parameters for calculating particle roundness using:
#@markdown - **Zheng & Hryciw (2015) algorithm**: Corner detection method with adjustable tolerance
#@markdown - **Vangla et al. (2018) smoothing**: Automated contour processing alternative

#@markdown **Key Parameters:**
#@markdown <ul>
#@markdown <li>🎚️ <b>α (Smoothing)</b>: Controls digital noise removal (0.01-0.1 typical)
#@markdown <li>⚖️ <b>T/R (Tolerance)</b>: Sets maximum deviation for corner circles (0.98-0.99 ideal)
#@markdown <li>🔄 <b>Vangla Smoothing</b>: Automatically removes digital noise from the contour (disable α when active)
#@markdown </ul>

from ipywidgets import Button, HBox, VBox, IntSlider, FloatSlider, Output, Checkbox
from IPython.display import display, clear_output

grains = {i: grain_data for i, (grain_id, grain_data) in enumerate(filtered_grains.items())}

# Create interactive widgets
id_slider = IntSlider(min=0, max=len(grains)-1, step=1, value=0, description='ID')
factor_slider = FloatSlider(min=0.98, max=1.0, step=0.001, value=0.985, description='T/R', readout_format='.4f')
span_slider = FloatSlider(min=0.001, max=0.25, step=0.002, value=0.04, description='α', readout_format='.3f')
smoothing_checkbox = Checkbox(value=False, description='Use Vangla Smoothing', indent=False)

# Function to update the sliders
def update_slider(slider, step):
    slider.value = round(slider.value + step, 3)

# Create buttons to move sliders in steps
id_up = Button(description='ID +1')
id_down = Button(description='ID -1')
factor_up = Button(description='T/R +0.001')
factor_down = Button(description='T/R -0.001')
span_up = Button(description='α +0.002')
span_down = Button(description='α -0.002')

# Link buttons to update functions
id_up.on_click(lambda b: update_slider(id_slider, 1))
id_down.on_click(lambda b: update_slider(id_slider, -1))
factor_up.on_click(lambda b: update_slider(factor_slider, 0.001))
factor_down.on_click(lambda b: update_slider(factor_slider, -0.001))
span_up.on_click(lambda b: update_slider(span_slider, 0.002))
span_down.on_click(lambda b: update_slider(span_slider, -0.002))

# Arrange buttons and sliders in new layout
controls = VBox([
    HBox([smoothing_checkbox]),
    HBox([id_down, id_slider, id_up]),
    HBox([factor_down, factor_slider, factor_up]),
    HBox([span_down, span_slider, span_up])
])

# Create an output widget for the plot
out = Output()
min_points=3
# Display the combined layout
display(controls, out)

def interactive_plot(id, factor, span, use_vangla):
    grain_data = grains[id]
    try:
        metrics = compute_shape_metrics(
            grain_data["grain_bin_rs"],
            PCD_Target,
            min_points,
            factor,
            use_vangla_smoothing=use_vangla,
            span=span
        )
        if metrics is None:
            print("compute_shape_metrics returned None")
            return
        AR, Cx, Sp, SAGI, de, dfmax, dfmin, dcir, Roundness, R_image = metrics
    except KeyError as e:
        print(f"KeyError: {e} is missing in grain_data")
        return
    with out:
        clear_output(wait=False)
        plot_grain(grain_data, AR, Cx, Sp, SAGI, de, dfmax, dfmin, dcir, Roundness, R_image)

# Update visibility of alpha slider based on checkbox
def update_slider_visibility(change):
    span_slider.layout.visibility = 'visible' if not change['new'] else 'hidden'
    span_up.layout.visibility = 'visible' if not change['new'] else 'hidden'
    span_down.layout.visibility = 'visible' if not change['new'] else 'hidden'

smoothing_checkbox.observe(update_slider_visibility, names='value')

# Call the interactive plot function when any value changes
def update_plot(*args):
    interactive_plot(
        id_slider.value,
        factor_slider.value,
        span_slider.value,
        smoothing_checkbox.value
    )

# Set observers for all interactive components
id_slider.observe(update_plot, 'value')
factor_slider.observe(update_plot, 'value')
span_slider.observe(update_plot, 'value')
smoothing_checkbox.observe(update_plot, 'value')

# Initial plot
update_plot()

In [ ]:
#@title **8. Calculate the metrics of all grains with the selected parameters for roundness calculation using the adaptation of the algorithm of Zheng & Hryciw 2015**.
#@markdown Enter the selected parameters:
from shape_metrics import compute_metrics_for_all_keys
#@markdown Use Vangla et al., (2018) Smooth? ⬇️
use_vangla_smoothing = True  #@param {type:"boolean", label:"α (span)"}
#@markdown smooth parameter, α: ⬇️
span = 0.04  #@param {type:"number", label:"α (span)"}
#@markdown factor, T/R ⬇️
factor = 0.983  #@param {type:"number", label:"T/R (factor)"}
#@markdown minimum points: ⬇️
min_points = 3  #@param {type:"number", label:"Minimum Points (min_points)"}
# alpha ={}
compute_metrics_for_all_keys(grains, PCD_Target, min_points, factor, use_vangla_smoothing, span)

from data_export import export_to_excel, shape_calculation, DATA_DIR, save_data_as_zip, plot_granulometric_curve, plot_density_curves

export_to_excel(material_name,grains)
gc.collect()
save_data_as_zip(material_name)

try:
    output_zip_filename = f"{material_name}.zip"
    files.download(output_zip_filename)
    print(f"Download initiated for {output_zip_filename}")
except Exception as e:
    print(f"Download failed: {e}")
    print(f"If you're running this in a notebook environment other than Google Colab, you may need to download the file manually.")


## Explanation of Excel Results

The Excel file contains comprehensive grain morphology analysis results:

### **Identification and Images**
- **`grain_id`**: Unique numerical identifier for each grain  
- **`grain_rs`**: Segmented grain  
- **`R_image`**: Processed image with analytical overlays

### **Size Measurements (pixels)**
- **`de`**: Equivalent diameter  
  - Represents the diameter of a circle with the same area as the projected area of the grain.  
   
  $$d_e = 2\sqrt{\frac{A}{\pi}}$$  
  *where $A$ is the grain's projected area.*

- **`dcir`**: Circumscribed circle diameter  
  - Represents the diameter of the smallest circle that fully encloses the grain.  
   
  $$d_{circ} = 2r_{circ}$$  
  *where $r_{circ}$ is the radius of the smallest enclosing circle.*

- **`minf`**: Minimum Feret diameter  
  - The smallest distance between two parallel tangents touching the grain's projection (often viewed as the grain's "thickness").  
   
  $$d_{Fmin}$$  
  *where $d_{Fmin}$ is the minimum Feret distance across the grain.*

- **`maxf`**: Maximum Feret diameter  
  - The largest distance between two parallel tangents touching the grain's projection (often viewed as the grain's "length").  
   
  $$d_{Fmax}$$  
  *where $d_{Fmax}$ is the maximum Feret distance across the grain.*

### **Shape Descriptors**

- **`Roundness`**: Wadell Roundness  
  - Quantifies how rounded the grain's corners are relative to its inscribed radius. Higher values indicate more rounded grains.  
   
  $$R = \frac{1}{N}\sum_{i=1}^N \frac{r_i}{r_{ins}}$$  
  *where $N$ is the total number of corners measured, $r_i$ are the corner radii, and $r_{ins}$ is the radius of the largest inscribed circle.*

- **`AR`**: Aspect Ratio  
  - Ratio of the smallest dimension to the largest dimension, indicating how elongated or flattened a grain is.  
   
  $$AR = \frac{d_{min}}{d_{max}}$$  
  *where $d_{min}$ is the smallest characteristic dimension (e.g., minimum Feret) and $d_{max}$ is the largest dimension (e.g., maximum Feret).*

- **`Cx`**: Convexity  
  - Compares the actual grain area to its convex hull area. Lower values indicate more concave or irregular shapes.  
   
  $$Cx = \frac{A}{A + B}$$  
  *where $A$ is the grain's projected area and $B$ is the extra area between the convex hull and the actual particle boundary.*

- **`Sp`**: Perimeter Sphericity  
  - Relates the grain's perimeter to that of a circle with the same area, indicating how close the shape is to being perfectly circular.  
   
  $$Sp = \frac{2\sqrt{\pi A}}{P}$$  
  *where $A$ is the grain's projected are and $P$ is the perimeter of the grain.*

- **`SAGI`**: Shape-Angularity Group Indicator  
  - Combines Aspect Ratio (AR), Convexity (Cx), and Sphericity (S) to classify particle angularity. Higher SAGI values indicate more angular grains (Altuhafi, Coop, & Georgiannou, 2016).  
  - Calculated as:  
    $$ \text{SAGI} = 5.4(1 - \text{AR}) - 67.8(1 - \text{Cx}) - 77.9(1 - \text{Sp}) $$  
  - **Classification Ranges**:  
    - **Rounded**: SAGI < 10.0  
    - **Subrounded**: 10.0 ≤ SAGI < 11.0  
    - **Subangular**: 11.0 ≤ SAGI < 12.0  
    - **Angular**: SAGI ≥ 12.0  
---
### **Notes on Units**
- Pixel-based measurements require scale conversion.
- Shape descriptors are dimensionless ratios.

The methods and calculations are based on:

- **Altuhafi, F. N., Coop, M. R., & Georgiannou, V. N.** (2016). Effect of Particle Shape on the Mechanical Behavior of Natural Sands. *Journal of Geotechnical and Geoenvironmental Engineering*, *142*(12), 04016071. https://doi.org/10.1061/(ASCE)GT.1943-5606.0001569  

- **Zheng, J., & Hryciw, R. D. (2015).** Traditional soil particle sphericity, roundness, and surface roughness by computational geometry. *Géotechnique*, 65(6), 494-506. [DOI: 10.1680/geot.14.P.192](https://doi.org/10.1680/geot.14.P.192)
- **Vangla, P., Roy, N., & Gali, M. L. (2018).** Image based shape characterization of granular materials and its effect on kinematics of particle motion. *Granular Matter*, 20(6). [DOI: 10.1007/s10035-017-0776-8](https://doi.org/10.1007/s10035-017-0776-8)

---

**Tip**: To convert pixel units to actual measurements, you need to know the scale of your images (e.g., pixels per millimeter). Once calibrated, multiply pixel measurements by the scale factor to get real-world dimensions.

**Understanding these parameters helps assess grain morphology and predict granular material behavior in geotechnical applications.**